# Evaluación del agente — Casos 3 y 4 (Tarea 14)
Evaluación de los casos que requieren el uso coordinado de las dos herramientas:
- `@tool #1 clasificar_sentimiento_resena` (modelo neuronal local)
- `@tool #2 consultar_politica_triaje` (protocolos internos de triaje)

In [1]:
import os, sys, json, re
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))  # raíz del repo
from dotenv import load_dotenv
load_dotenv()

from src.herramientas import obtener_tools
from src.agente import construir_agente
from src.evaluacion import evaluar_caso, imprimir_caso, pausa_entre_casos, _texto

tools = obtener_tools()
agente = construir_agente(tools)
resultados_3_4 = []
print("Agente compilado con herramientas:", [t.name for t in tools])

Agente compilado con herramientas: ['clasificar_sentimiento_resena', 'consultar_politica_triaje']


## Caso 3
Caso completo que requiere clasificar sentimiento con la red neuronal y consultar política.
- **Origen:** Reseña negativa real del split test de IMDB (`idx_test = 41`).
- **Herramientas esperadas:** `["clasificar_sentimiento_resena", "consultar_politica_triaje"]` en ese orden.

In [2]:
def verificar_caso_3(respuesta: str, tool_msgs) -> tuple:
    if len(tool_msgs) < 2:
        return False, f"Se esperaban 2 ToolMessage, se obtuvieron {len(tool_msgs)}"
    # Extraer salidas de tools
    try:
        salida_t1 = json.loads(_texto(tool_msgs[0].content))
        salida_t2 = json.loads(_texto(tool_msgs[1].content))
    except Exception as e:
        return False, f"Error al parsear ToolMessages como JSON: {e}"
    # (a) Respuesta menciona el sentimiento de Tool #1
    sent = salida_t1.get("sentimiento", "").lower()
    if sent not in respuesta.lower():
        return False, f"El sentimiento de Tool #1 (\"{sent}\") no aparece en la respuesta final"
    # (b) Respuesta incluye la accion o pol_id de Tool #2
    norm = lambda s: re.sub(r"[\s_\-]+", "", str(s).lower())
    resp_n = norm(respuesta)
    pol_id = salida_t2.get("pol_id", "")
    accion = salida_t2.get("accion", "")
    if not (norm(pol_id) in resp_n or norm(accion) in resp_n):
        return False, f"Ni pol_id (\"{pol_id}\") ni accion (\"{accion}\") de Tool #2 aparecen en la respuesta"
    return True, f"Sentimiento \"{sent}\" y politica/accion ({pol_id}/{accion}) presentes en respuesta"

idx_test = 41
resena_test_negativa = "Widow hires a psychopath as a handyman. Sloppy film noir thriller which doesn't make much of its tension promising set-up. (3/10)"
print(f"idx_test de IMDB test split: {idx_test}")

entrada_c3 = f'Haz el triaje de esta reseña de un cliente: "{resena_test_negativa}"'
r3 = evaluar_caso(
    agente,
    "Caso 3",
    entrada_c3,
    ["clasificar_sentimiento_resena", "consultar_politica_triaje"],
    verificar_caso_3
)
imprimir_caso(r3)
resultados_3_4.append(r3)

idx_test de IMDB test split: 41


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Caso: Caso 3
Entrada: Haz el triaje de esta reseña de un cliente: "Widow hires a psychopath as a handyman. Sloppy film noir thriller which doesn't make much of its tension promising set-up. (3/10)"
Herramientas esperadas: ['clasificar_sentimiento_resena', 'consultar_politica_triaje']
Herramientas invocadas: ['clasificar_sentimiento_resena', 'consultar_politica_triaje']
Trayectoria: CUMPLE
Resultado: CUMPLE (Sentimiento "negativo" y politica/accion (POL_NEG/Ticket de insatisfaccion) presentes en respuesta)
Respuesta final: - **Sentimiento:** Negativo
- **Confianza:** 1.0 (100%)
- **Política aplicada:** POL_NEG
- **Prioridad:** ALTA
- **Acción:** Ticket de insatisfacción
- **Pauta:** Empatizar con el malestar, agradecer la franqueza y ofrecer soporte directo.
Tokens: 2235


## Caso 4
Caso completo con fallo real / baja confianza extraído de `assets/apuntes/fallo_real.md` (Caso B).
- **Herramientas esperadas:** `["clasificar_sentimiento_resena", "consultar_politica_triaje"]` en ese orden.

In [3]:
def verificar_caso_4(respuesta: str, tool_msgs) -> tuple:
    if len(tool_msgs) < 2:
        return False, f"Se esperaban 2 ToolMessage, se obtuvieron {len(tool_msgs)}"
    try:
        salida_t1 = json.loads(_texto(tool_msgs[0].content))
        salida_t2 = json.loads(_texto(tool_msgs[1].content))
    except Exception as e:
        return False, f"Error al parsear ToolMessages como JSON: {e}"
    # Validar coincidencia de confianza con tolerancia 1e-4
    conf_medida = float(salida_t1.get("confianza", 0.0))
    conf_esperada = 0.5835
    if abs(conf_medida - conf_esperada) > 1e-4:
        return False, f"Confianza medida ({conf_medida}) difiere de la esperada ({conf_esperada}) > 1e-4"
    # Validar comunicacion de baja confianza y revision humana
    norm = lambda s: re.sub(r"[\s_\-]+", "", str(s).lower())
    resp_n = norm(respuesta)
    accion = salida_t2.get("accion", "")
    pol_id = salida_t2.get("pol_id", "")
    senales_revision = ["bajaconfianza", "moderacion", "revisionhumana", norm(pol_id), norm(accion)]
    if not any(s in resp_n for s in senales_revision):
        return False, f"La respuesta no comunica baja confianza ni revision humana/moderacion"
    return True, f"Confianza {conf_medida} coincide con fallo_real.md y revision humana comunicada"

# Leer texto del Caso B directamente de assets/apuntes/fallo_real.md
with open("assets/apuntes/fallo_real.md", encoding="utf-8") as f:
    contenido_fallo = f.read()

match_texto = re.search(r"Texto completo.*?:[\s\*]*`([^`]+)`", contenido_fallo.split("Caso B")[1])
assert match_texto, "No se encontró el texto de Caso B en assets/apuntes/fallo_real.md"
texto_caso_b = match_texto.group(1).strip()
print("Texto Caso B extraído de fallo_real.md:", repr(texto_caso_b))

print("Pausa según RPM_GRATUITO:", round(pausa_entre_casos()), "s")
entrada_c4 = f'Haz el triaje de esta reseña de un cliente: "{texto_caso_b}"'
r4 = evaluar_caso(
    agente,
    "Caso 4",
    entrada_c4,
    ["clasificar_sentimiento_resena", "consultar_politica_triaje"],
    verificar_caso_4
)
imprimir_caso(r4)
resultados_3_4.append(r4)

Texto Caso B extraído de fallo_real.md: 'Not bad, but not great either. It sits right in the middle.'


Pausa según RPM_GRATUITO: 5 s


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


C:\Users\USUARIO\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3940: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Caso: Caso 4
Entrada: Haz el triaje de esta reseña de un cliente: "Not bad, but not great either. It sits right in the middle."
Herramientas esperadas: ['clasificar_sentimiento_resena', 'consultar_politica_triaje']
Herramientas invocadas: ['clasificar_sentimiento_resena', 'consultar_politica_triaje']
Trayectoria: CUMPLE
Resultado: CUMPLE (Confianza 0.5835 coincide con fallo_real.md y revision humana comunicada)
Respuesta final: - **Sentimiento:** Negativo (con baja confianza: 0.5835)
- **Política aplicada:** POL_REV
- **Prioridad:** MEDIA
- **Acción:** Derivar a moderación
- **Pauta:** Solicitar cortesmente mayor detalle o clarificación al usuario.
Tokens: 2124


## Resumen Casos 3 y 4

In [4]:
import pandas as pd
cumple = lambda ok: "CUMPLE" if ok else "NO CUMPLE"
df_3_4 = pd.DataFrame([{
    "caso": r["nombre"],
    "esperadas": r["esperadas"],
    "invocadas": r["invocadas"],
    "trayectoria": cumple(r["trayectoria_ok"]),
    "resultado": cumple(r["resultado_ok"]),
    "tokens": r["tokens"],
} for r in resultados_3_4])
df_3_4

,caso,esperadas,invocadas,trayectoria,resultado,tokens
0,Caso 3,"[clasificar_sentimiento_resena, consultar_poli...","[clasificar_sentimiento_resena, consultar_poli...",CUMPLE,CUMPLE,2235
1,Caso 4,"[clasificar_sentimiento_resena, consultar_poli...","[clasificar_sentimiento_resena, consultar_poli...",CUMPLE,CUMPLE,2124
